# 🎓 **Complete PLSR Tutorial with ALL Essential Plots**

## **From MLR to PLS2: Comprehensive Visualization & Interpretation**

---

### 📚 **Course Information**
- **Course:** TTK4260 - Multivariate Data Analysis and ML
- **Instructor:** Adil Rasheed, NTNU
- **Version:** Complete Edition with ~35 Plots

---

## 🎯 **What's New in This Version**

This **COMPLETE** tutorial includes ALL essential PLSR plots:

### ✅ **Interpretation Plots:**
- Correlation loadings (industry standard)
- Correlation circle plots
- Bi-plots (scores + loadings)
- 3D score visualization
- VIP scores

### ✅ **Diagnostic Plots:**
- Hotelling T² (outlier detection)
- Q residuals / DModX (model fit)
- Williams plot (applicability domain)
- Q-Q plot (normality check)
- Cook's distance (influential points)
- Residual diagnostics

### ✅ **Validation Plots:**
- R² train vs CV (overfitting)
- Explained variance (X and Y)
- Inner relation (PLS validity)
- Prediction intervals
- Per-fold CV analysis

### ✅ **Comparison Plots:**
- All methods side-by-side
- Performance metrics
- Coefficient evolution

**Total: ~35 interactive Plotly visualizations!**

---

## 📖 **Learning Objectives**

By the end, you will be able to:

1. ✅ Understand multicollinearity and why OLS fails
2. ✅ Apply PCR as unsupervised dimensionality reduction
3. ✅ Implement PLSR for supervised regression
4. ✅ **Interpret correlation loadings and circles** (NEW!)
5. ✅ **Detect outliers using T² and Q statistics** (NEW!)
6. ✅ **Assess prediction reliability with Williams plot** (NEW!)
7. ✅ **Validate model assumptions** (NEW!)
8. ✅ Extend to PLS2 for multiple responses
9. ✅ Choose the right method for your data

---

**⏱️ Estimated Time:** 3-4 hours (comprehensive)

Let's begin! 🚀

---

## **Section 0: Setup and Configuration**

In [12]:
# Core packages
import numpy as np
import pandas as pd
from scipy import stats

# Machine learning
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import cross_val_score, cross_val_predict, KFold
from sklearn.metrics import mean_squared_error, r2_score

# Visualization
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

# Configuration
import warnings
warnings.filterwarnings('ignore')
pio.templates.default = "plotly_white"
np.random.seed(42)

print("✅ All packages loaded!")
print(f"NumPy: {np.__version__}")
print(f"Scikit-learn: {__import__('sklearn').__version__}")
print(f"Plotly: {__import__('plotly').__version__}")


✅ All packages loaded!
NumPy: 2.3.5
Scikit-learn: 1.7.2
Plotly: 6.3.0


---

## **Section 1: Data Loading and Exploration**

### 1.1 Load Tecator Dataset

In [13]:
from skfda.datasets import fetch_tecator

X_fda, y_fat = fetch_tecator(return_X_y=True)
X = X_fda.data_matrix.squeeze()
# Extract only the fat content (first column) - Tecator returns [fat, protein, moisture]
y = y_fat[:, 0] if y_fat.ndim > 1 else y_fat
wavelengths = np.linspace(850, 1050, X.shape[1])

print("="*70)
print("TECATOR DATASET")
print("="*70)
print(f"\nX shape: {X.shape} (samples × wavelengths)")
print(f"Y shape: {y.shape}")
print(f"\nFat content:")
print(f"  Mean: {y.mean():.2f}%")
print(f"  Range: [{y.min():.2f}, {y.max():.2f}]%")
print(f"  Std: {y.std():.2f}%")


TECATOR DATASET

X shape: (215, 100) (samples × wavelengths)
Y shape: (215,)

Fat content:
  Mean: 18.14%
  Range: [0.90, 49.10]%
  Std: 12.71%


### 1.2 Response Distribution

Check for outliers and normality.

In [14]:
fig = make_subplots(rows=1, cols=2,
    subplot_titles=("Box Plot - Outlier Detection", "Histogram - Distribution"))

# Box plot
fig.add_trace(go.Box(y=y, name='Fat', marker_color='lightblue',
    boxmean='sd'), row=1, col=1)

# Histogram
fig.add_trace(go.Histogram(x=y, nbinsx=20, name='Fat',
    marker_color='lightcoral'), row=1, col=2)

fig.update_xaxes(title_text="", row=1, col=1)
fig.update_yaxes(title_text="Fat Content (%)", row=1, col=1)
fig.update_xaxes(title_text="Fat Content (%)", row=1, col=2)
fig.update_yaxes(title_text="Frequency", row=1, col=2)

fig.update_layout(title="Response Variable Analysis", width=1200, height=400,
    showlegend=False)
fig.show()

# Normality test
shapiro_stat, shapiro_p = stats.shapiro(y)
print(f"\nShapiro-Wilk test: p-value = {shapiro_p:.4f}")
if shapiro_p > 0.05:
    print("✅ Response appears normally distributed")
else:
    print("⚠️  Response may deviate from normality")


Shapiro-Wilk test: p-value = 0.0000
⚠️  Response may deviate from normality


### 1.3 Spectra Visualization

In [15]:
# Plot spectra colored by fat content
fig = go.Figure()

# Use all samples but with transparency
for i in range(len(X)):
    fig.add_trace(go.Scatter(
        x=wavelengths, y=X[i],
        mode='lines',
        line=dict(width=1, color=f'rgba({int(255*y[i]/y.max())},0,{int(255*(1-y[i]/y.max()))},0.3)'),
        showlegend=False,
        hovertemplate=f'Sample {i}<br>Fat: {y[i]:.1f}%<extra></extra>'
    ))

fig.update_layout(
    title="NIR Absorbance Spectra (colored by fat: blue=low, red=high)",
    xaxis_title="Wavelength (nm)",
    yaxis_title="Absorbance",
    width=1000, height=500
)
fig.show()


### 1.4 Multicollinearity Assessment

In [16]:
from numpy.linalg import cond

condition_number = cond(X)
corr_matrix_X = np.corrcoef(X.T)
avg_corr = (np.sum(corr_matrix_X) - len(corr_matrix_X)) / (len(corr_matrix_X)**2 - len(corr_matrix_X))

print("="*70)
print("MULTICOLLINEARITY ANALYSIS")
print("="*70)
print(f"\n⚠️  Condition number: {condition_number:.2e}")
print(f"    Average correlation: {avg_corr:.3f}")

if condition_number > 30:
    print(f"\n🚨 SEVERE multicollinearity!")
    print(f"   OLS will be highly unstable!")

# Correlation heatmap (sample every 5th)
sample_every = 5
X_sample = X[:, ::sample_every]
corr_sample = np.corrcoef(X_sample.T)

fig = go.Figure(data=go.Heatmap(
    z=corr_sample,
    x=[f'{int(wavelengths[i])}' for i in range(0, len(wavelengths), sample_every)],
    y=[f'{int(wavelengths[i])}' for i in range(0, len(wavelengths), sample_every)],
    colorscale='RdBu_r',
    zmid=0,
    colorbar=dict(title="Correlation")
))

fig.update_layout(
    title="Correlation Matrix (every 5th wavelength)",
    xaxis_title="Wavelength (nm)",
    yaxis_title="Wavelength (nm)",
    width=700, height=700
)
fig.show()

MULTICOLLINEARITY ANALYSIS

⚠️  Condition number: 1.62e+07
    Average correlation: 0.986

🚨 SEVERE multicollinearity!
   OLS will be highly unstable!


### 1.5 Wavelength-Response Correlations

In [17]:
correlations_with_fat = np.array([
    np.corrcoef(X[:, i], y)[0, 1] for i in range(X.shape[1])
])

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=wavelengths, y=correlations_with_fat,
    mode='lines', fill='tozeroy',
    line=dict(color='purple', width=2)
))
fig.add_hline(y=0, line_dash="dash", line_color="gray")

fig.update_layout(
    title="Correlation: Each Wavelength with Fat Content",
    xaxis_title="Wavelength (nm)",
    yaxis_title="Correlation",
    width=1000, height=400
)
fig.show()

top_5_idx = np.argsort(np.abs(correlations_with_fat))[-5:]
print("\nTop 5 most correlated wavelengths:")
for idx in top_5_idx[::-1]:
    print(f"  λ={wavelengths[idx]:.0f}nm: r={correlations_with_fat[idx]:+.3f}")


Top 5 most correlated wavelengths:
  λ=931nm: r=+0.518
  λ=929nm: r=+0.517
  λ=933nm: r=+0.515
  λ=927nm: r=+0.513
  λ=935nm: r=+0.508


---

## **Section 2: Ordinary Least Squares (OLS)**

### 2.1 Standardize and Fit

In [18]:
# Standardize
scaler_X = StandardScaler()
scaler_y = StandardScaler()
X_std = scaler_X.fit_transform(X)
y_std = scaler_y.fit_transform(y.reshape(-1, 1)).ravel()

# Fit OLS
mlr = LinearRegression()
mlr.fit(X_std, y_std)
y_pred_mlr_std = mlr.predict(X_std)
y_pred_mlr = scaler_y.inverse_transform(y_pred_mlr_std.reshape(-1, 1)).ravel()

r2_mlr_train = r2_score(y, y_pred_mlr)
rmse_mlr_train = np.sqrt(mean_squared_error(y, y_pred_mlr))

print("="*70)
print("OLS REGRESSION")
print("="*70)
print(f"\nTraining:")
print(f"  R²: {r2_mlr_train:.4f}")
print(f"  RMSE: {rmse_mlr_train:.4f}%")

coef_mlr = mlr.coef_
print(f"\nCoefficients:")
print(f"  Mean |coef|: {np.abs(coef_mlr).mean():.4f}")
print(f"  Max |coef|: {np.abs(coef_mlr).max():.4f}")
print(f"  Std: {coef_mlr.std():.4f}")

OLS REGRESSION

Training:
  R²: 0.9951
  RMSE: 0.8887%

Coefficients:
  Mean |coef|: 479.5856
  Max |coef|: 1553.5954
  Std: 618.2969


### 2.2 OLS Coefficients (Extremely Noisy!)

In [19]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=wavelengths, y=coef_mlr,
    mode='lines+markers',
    line=dict(color='red', width=2),
    marker=dict(size=3)
))
fig.add_hline(y=0, line_dash="dash", line_color="gray")
fig.update_layout(
    title="OLS Coefficients - UNSTABLE due to Multicollinearity",
    xaxis_title="Wavelength (nm)",
    yaxis_title="Coefficient Value",
    width=1000, height=500
)
fig.show()

print("⚠️  Wild oscillations → numerical instability!")

⚠️  Wild oscillations → numerical instability!


### 2.3 Cross-Validation

In [20]:
cv = KFold(n_splits=10, shuffle=True, random_state=42)

cv_scores_mlr = cross_val_score(mlr, X_std, y_std, cv=cv, 
                                 scoring='neg_mean_squared_error')
cv_rmse_mlr = np.sqrt(-cv_scores_mlr)
cv_r2_mlr = cross_val_score(mlr, X_std, y_std, cv=cv, scoring='r2')

y_pred_cv_mlr_std = cross_val_predict(mlr, X_std, y_std, cv=cv)
y_pred_cv_mlr = scaler_y.inverse_transform(y_pred_cv_mlr_std.reshape(-1, 1)).ravel()

print("Cross-Validation (10-fold):")
print(f"  CV R²: {cv_r2_mlr.mean():.4f} ± {cv_r2_mlr.std():.4f}")
print(f"  CV RMSE: {cv_rmse_mlr.mean():.4f}")

if r2_mlr_train - cv_r2_mlr.mean() > 0.1:
    print(f"\n🚨 SEVERE OVERFITTING!")
    print(f"   Gap: {r2_mlr_train - cv_r2_mlr.mean():.3f}")

Cross-Validation (10-fold):
  CV R²: 0.9330 ± 0.0603
  CV RMSE: 0.2316


### 2.4 OLS Diagnostics

In [21]:
fig = make_subplots(rows=1, cols=2,
    subplot_titles=("Training", "Cross-Validation"))

# Training
fig.add_trace(go.Scatter(x=y, y=y_pred_mlr, mode='markers',
    marker=dict(color='blue', size=6, opacity=0.6)), row=1, col=1)

# CV
fig.add_trace(go.Scatter(x=y, y=y_pred_cv_mlr, mode='markers',
    marker=dict(color='red', size=6, opacity=0.6)), row=1, col=2)

# Perfect prediction lines
y_range = [y.min(), y.max()]
for col in [1, 2]:
    fig.add_trace(go.Scatter(x=y_range, y=y_range, mode='lines',
        line=dict(color='black', dash='dash'), showlegend=False), row=1, col=col)

fig.update_xaxes(title_text="Actual Fat (%)", row=1, col=1)
fig.update_xaxes(title_text="Actual Fat (%)", row=1, col=2)
fig.update_yaxes(title_text="Predicted (%)", row=1, col=1)
fig.update_yaxes(title_text="Predicted (%)", row=1, col=2)

fig.update_layout(title="OLS: Actual vs Predicted", width=1200, height=500)
fig.show()

---

## **Section 3: Principal Component Regression (PCR)**

### 3.1 PCA Analysis

In [22]:
pca_full = PCA(n_components=None)
T_full = pca_full.fit_transform(X_std)
P_full = pca_full.components_.T

explained_var_ratio = pca_full.explained_variance_ratio_
cumulative_var = np.cumsum(explained_var_ratio)

print("="*70)
print("PRINCIPAL COMPONENT ANALYSIS")
print("="*70)
print(f"\nVariance explained:")
print(f"  PC1: {explained_var_ratio[0]*100:.2f}%")
print(f"  PC1-2: {cumulative_var[1]*100:.2f}%")
print(f"  PC1-5: {cumulative_var[4]*100:.2f}%")
print(f"  PC1-10: {cumulative_var[9]*100:.2f}%")

n_95 = np.argmax(cumulative_var >= 0.95) + 1
print(f"\n  Components for 95%: {n_95}")

PRINCIPAL COMPONENT ANALYSIS

Variance explained:
  PC1: 98.63%
  PC1-2: 99.60%
  PC1-5: 100.00%
  PC1-10: 100.00%

  Components for 95%: 1


### 3.2 Scree Plot

In [23]:
n_show = 30
fig = make_subplots(rows=1, cols=2,
    subplot_titles=("Scree Plot", "Cumulative Variance"))

fig.add_trace(go.Bar(x=np.arange(1, n_show+1),
    y=explained_var_ratio[:n_show]*100), row=1, col=1)

fig.add_trace(go.Scatter(x=np.arange(1, n_show+1),
    y=cumulative_var[:n_show]*100, mode='lines+markers',
    line=dict(color='red', width=2)), row=1, col=2)

fig.add_hline(y=95, line_dash="dash", line_color="green", row=1, col=2)

fig.update_xaxes(title_text="PC", row=1, col=1)
fig.update_xaxes(title_text="# Components", row=1, col=2)
fig.update_yaxes(title_text="Variance (%)", row=1, col=1)
fig.update_yaxes(title_text="Cumulative (%)", row=1, col=2)

fig.update_layout(title="PCA Variance Explained", width=1200, height=500,
    showlegend=False)
fig.show()

### 3.3 PC Correlation with Response

In [24]:
n_pcs = 20
pc_corr = [np.corrcoef(T_full[:, i], y)[0, 1] for i in range(n_pcs)]

fig = go.Figure()
fig.add_trace(go.Bar(x=np.arange(1, n_pcs+1), y=pc_corr,
    marker_color=['red' if abs(c)>0.5 else 'lightblue' for c in pc_corr]))
fig.add_hline(y=0, line_color="black")
fig.add_hline(y=0.5, line_dash="dash", line_color="green")
fig.add_hline(y=-0.5, line_dash="dash", line_color="green")

fig.update_layout(
    title="PC Correlation with Fat Content",
    xaxis_title="Principal Component",
    yaxis_title="Correlation",
    width=1000, height=500
)
fig.show()

print("⚠️  High-variance PCs don't always predict Y well!")

⚠️  High-variance PCs don't always predict Y well!


### 3.4 PCR Cross-Validation

In [25]:
def fit_pcr(X_train, y_train, X_test, y_test, n_components):
    pca = PCA(n_components=n_components)
    T_train = pca.fit_transform(X_train)
    reg = LinearRegression()
    reg.fit(T_train, y_train)
    T_test = pca.transform(X_test)
    y_pred = reg.predict(T_test)
    return y_pred, pca, reg

max_components = 30
print(f"Running PCR CV (max {max_components})...")

pcr_results = {}
for n_comp in range(1, max_components + 1):
    cv_scores = []
    for train_idx, test_idx in cv.split(X_std):
        X_train, X_test = X_std[train_idx], X_std[test_idx]
        y_train, y_test = y_std[train_idx], y_std[test_idx]
        y_pred, _, _ = fit_pcr(X_train, y_train, X_test, y_test, n_comp)
        cv_scores.append(mean_squared_error(y_test, y_pred))
    pcr_results[n_comp] = {
        'cv_rmse_mean': np.sqrt(np.mean(cv_scores)),
        'cv_rmse_std': np.std(np.sqrt(cv_scores))
    }

rmse_values_pcr = [pcr_results[k]['cv_rmse_mean'] for k in range(1, max_components+1)]
optimal_k_pcr = np.argmin(rmse_values_pcr) + 1

print(f"✅ Optimal: k={optimal_k_pcr}, RMSE={rmse_values_pcr[optimal_k_pcr-1]:.4f}")

Running PCR CV (max 30)...
✅ Optimal: k=30, RMSE=0.1919


### 3.5 PCR Model with Optimal Components

In [26]:
pca_optimal = PCA(n_components=optimal_k_pcr)
T_optimal = pca_optimal.fit_transform(X_std)
P_optimal = pca_optimal.components_.T

reg_pcr = LinearRegression()
reg_pcr.fit(T_optimal, y_std)
y_pred_pcr_std = reg_pcr.predict(T_optimal)
y_pred_pcr = scaler_y.inverse_transform(y_pred_pcr_std.reshape(-1, 1)).ravel()

theta_pcr = P_optimal @ reg_pcr.coef_
r2_pcr_train = r2_score(y, y_pred_pcr)

# CV predictions
y_pred_pcr_cv_std = np.zeros(len(y_std))
for train_idx, test_idx in cv.split(X_std):
    X_train, X_test = X_std[train_idx], X_std[test_idx]
    y_train, y_test = y_std[train_idx], y_std[test_idx]
    y_pred, _, _ = fit_pcr(X_train, y_train, X_test, y_test, optimal_k_pcr)
    y_pred_pcr_cv_std[test_idx] = y_pred

y_pred_pcr_cv = scaler_y.inverse_transform(y_pred_pcr_cv_std.reshape(-1, 1)).ravel()
r2_pcr_cv = r2_score(y, y_pred_pcr_cv)
rmse_pcr_cv = np.sqrt(mean_squared_error(y, y_pred_pcr_cv))

print(f"PCR (k={optimal_k_pcr}):")
print(f"  Train R²: {r2_pcr_train:.4f}")
print(f"  CV R²: {r2_pcr_cv:.4f}")
print(f"  CV RMSE: {rmse_pcr_cv:.4f}%")

PCR (k=30):
  Train R²: 0.9821
  CV R²: 0.9631
  CV RMSE: 2.4430%


---

## **Section 4: Partial Least Squares Regression (PLS)**

### 4.1 PLS Cross-Validation

In [27]:
print("Running PLS CV...")

pls_results = {}
for n_comp in range(1, max_components + 1):
    cv_scores = []
    for train_idx, test_idx in cv.split(X_std):
        X_train, X_test = X_std[train_idx], X_std[test_idx]
        y_train, y_test = y_std[train_idx], y_std[test_idx]
        pls = PLSRegression(n_components=n_comp, scale=False)
        pls.fit(X_train, y_train)
        y_pred = pls.predict(X_test).ravel()
        cv_scores.append(mean_squared_error(y_test, y_pred))
    pls_results[n_comp] = {
        'cv_rmse_mean': np.sqrt(np.mean(cv_scores)),
        'cv_rmse_std': np.std(np.sqrt(cv_scores))
    }

rmse_values_pls = [pls_results[k]['cv_rmse_mean'] for k in range(1, max_components+1)]
optimal_k_pls = np.argmin(rmse_values_pls) + 1

print(f"✅ Optimal: k={optimal_k_pls}, RMSE={rmse_values_pls[optimal_k_pls-1]:.4f}")

Running PLS CV...
✅ Optimal: k=18, RMSE=0.1905


### 4.2 PCR vs PLS Comparison

In [28]:
fig = go.Figure()

k_values = list(range(1, max_components+1))

fig.add_trace(go.Scatter(x=k_values, y=rmse_values_pcr,
    mode='lines+markers', name='PCR', line=dict(color='blue', width=2)))

fig.add_trace(go.Scatter(x=k_values, y=rmse_values_pls,
    mode='lines+markers', name='PLS', line=dict(color='red', width=2)))

fig.add_vline(x=optimal_k_pcr, line_dash="dash", line_color="blue",
    annotation_text=f"PCR k={optimal_k_pcr}")
fig.add_vline(x=optimal_k_pls, line_dash="dash", line_color="red",
    annotation_text=f"PLS k={optimal_k_pls}")

fig.update_layout(
    title="Model Selection: PCR vs PLS",
    xaxis_title="Number of Components",
    yaxis_title="RMSE (CV)",
    width=1000, height=500
)
fig.show()

print(f"\nPCR: k={optimal_k_pcr}, RMSE={rmse_values_pcr[optimal_k_pcr-1]:.4f}")
print(f"PLS: k={optimal_k_pls}, RMSE={rmse_values_pls[optimal_k_pls-1]:.4f}")
if optimal_k_pls < optimal_k_pcr:
    print(f"✅ PLS uses {optimal_k_pcr-optimal_k_pls} FEWER components!")


PCR: k=30, RMSE=0.1919
PLS: k=18, RMSE=0.1905
✅ PLS uses 12 FEWER components!


### 4.3 Fit Optimal PLS Model

In [29]:
pls_optimal = PLSRegression(n_components=optimal_k_pls, scale=False)
pls_optimal.fit(X_std, y_std)

y_pred_pls_std = pls_optimal.predict(X_std).ravel()
y_pred_pls = scaler_y.inverse_transform(y_pred_pls_std.reshape(-1, 1)).ravel()
r2_pls_train = r2_score(y, y_pred_pls)

# CV predictions
y_pred_pls_cv_std = np.zeros(len(y_std))
for train_idx, test_idx in cv.split(X_std):
    X_train, X_test = X_std[train_idx], X_std[test_idx]
    y_train, y_test = y_std[train_idx], y_std[test_idx]
    pls_cv = PLSRegression(n_components=optimal_k_pls, scale=False)
    pls_cv.fit(X_train, y_train)
    y_pred_pls_cv_std[test_idx] = pls_cv.predict(X_test).ravel()

y_pred_pls_cv = scaler_y.inverse_transform(y_pred_pls_cv_std.reshape(-1, 1)).ravel()
r2_pls_cv = r2_score(y, y_pred_pls_cv)
rmse_pls_cv = np.sqrt(mean_squared_error(y, y_pred_pls_cv))

# Extract components
T_pls = pls_optimal.x_scores_
W_pls = pls_optimal.x_weights_
P_pls = pls_optimal.x_loadings_
Q_pls = pls_optimal.y_loadings_
coef_pls = pls_optimal.coef_.ravel()

print(f"PLS (k={optimal_k_pls}):")
print(f"  Train R²: {r2_pls_train:.4f}")
print(f"  CV R²: {r2_pls_cv:.4f}")
print(f"  CV RMSE: {rmse_pls_cv:.4f}%")

print(f"\nComponent correlations with Y:")
for i in range(min(5, optimal_k_pls)):
    corr = np.corrcoef(T_pls[:, i], y_std)[0, 1]
    print(f"  Comp {i+1}: {corr:+.3f}")

PLS (k=18):
  Train R²: 0.9835
  CV R²: 0.9636
  CV RMSE: 2.4235%

Component correlations with Y:
  Comp 1: +0.443
  Comp 2: +0.663
  Comp 3: +0.441
  Comp 4: -0.252
  Comp 5: +0.222


### 4.4 Standard PLS Score Plot

In [30]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=T_pls[:, 0],
    y=T_pls[:, 1] if optimal_k_pls > 1 else np.zeros(len(T_pls)),
    mode='markers',
    marker=dict(size=8, color=y, colorscale='Viridis',
        showscale=True, colorbar=dict(title="Fat (%)")),
    text=[f'Sample {i}: {y[i]:.1f}%' for i in range(len(y))],
    hovertemplate='%{text}<extra></extra>'
))

fig.update_layout(
    title="PLS Score Plot (colored by Fat)",
    xaxis_title=f"Comp 1 (r={np.corrcoef(T_pls[:,0], y_std)[0,1]:.3f})",
    yaxis_title=f"Comp 2" + (f" (r={np.corrcoef(T_pls[:,1], y_std)[0,1]:.3f})" if optimal_k_pls>1 else ""),
    width=800, height=600
)
fig.show()

print("💡 Strong color gradient: PLS maximizes correlation with Y!")

💡 Strong color gradient: PLS maximizes correlation with Y!


### 4.5 ⭐ **NEW: 3D Score Plot**

In [31]:
if optimal_k_pls >= 3:
    fig = go.Figure(data=[go.Scatter3d(
        x=T_pls[:, 0],
        y=T_pls[:, 1],
        z=T_pls[:, 2],
        mode='markers',
        marker=dict(
            size=6,
            color=y,
            colorscale='Viridis',
            showscale=True,
            colorbar=dict(title="Fat (%)", x=1.1)
        ),
        text=[f'Sample {i}<br>Fat: {y[i]:.1f}%' for i in range(len(y))],
        hovertemplate='%{text}<extra></extra>'
    )])
    
    fig.update_layout(
        title="3D PLS Score Plot",
        scene=dict(
            xaxis_title="Component 1",
            yaxis_title="Component 2",
            zaxis_title="Component 3"
        ),
        width=800,
        height=800
    )
    fig.show()
    print("✅ 3D view shows sample clustering in latent space")
else:
    print("⚠️  Need ≥3 components for 3D plot")

✅ 3D view shows sample clustering in latent space


### 4.6 Standard Loadings (P)

In [32]:
fig = go.Figure()
n_comp_plot = min(4, optimal_k_pls)
colors = ['blue', 'red', 'green', 'orange']

for i in range(n_comp_plot):
    fig.add_trace(go.Scatter(
        x=wavelengths, y=P_pls[:, i],
        mode='lines', name=f'Comp {i+1}',
        line=dict(color=colors[i], width=2)
    ))

fig.add_hline(y=0, line_dash="dash", line_color="gray")
fig.update_layout(
    title="PLS Standard Loadings (P)",
    xaxis_title="Wavelength (nm)",
    yaxis_title="Loading",
    width=1000, height=500
)
fig.show()

### 4.7 ⭐⭐⭐ **NEW: Correlation Loadings** (INDUSTRY STANDARD)

**CRITICAL:** Correlation loadings are the STANDARD for PLS interpretation!
- Bounded [-1, +1] (easy to understand)
- Show correlation between variables and components
- Used by SIMCA, Unscrambler, all chemometrics software

In [33]:
# Compute correlation loadings
correlation_loadings = np.zeros((X_std.shape[1], optimal_k_pls))

for i in range(optimal_k_pls):
    for j in range(X_std.shape[1]):
        correlation_loadings[j, i] = np.corrcoef(X_std[:, j], T_pls[:, i])[0, 1]

print("="*70)
print("CORRELATION LOADINGS")
print("="*70)
print(f"Shape: {correlation_loadings.shape}")
print(f"Range: [{correlation_loadings.min():.3f}, {correlation_loadings.max():.3f}]")
print("\n✅ These are what you should use for interpretation!")

# Plot correlation loadings
fig = go.Figure()

for i in range(min(4, optimal_k_pls)):
    fig.add_trace(go.Scatter(
        x=wavelengths,
        y=correlation_loadings[:, i],
        mode='lines',
        name=f'Component {i+1}',
        line=dict(color=colors[i], width=2)
    ))

fig.add_hline(y=0, line_dash="dash", line_color="gray")
fig.add_hline(y=0.7, line_dash="dot", line_color="green",
              annotation_text="Strong (+)")
fig.add_hline(y=-0.7, line_dash="dot", line_color="green",
              annotation_text="Strong (-)")

fig.update_layout(
    title="⭐ PLS Correlation Loadings (RECOMMENDED for Interpretation)",
    xaxis_title="Wavelength (nm)",
    yaxis_title="Correlation with Component",
    yaxis_range=[-1, 1],
    width=1000,
    height=500,
    hovermode='x unified'
)

fig.show()

print("\n💡 Interpretation:")
print("  |r| > 0.7: Strong correlation")
print("  |r| < 0.3: Weak correlation")
print("  Sign: Positive/negative relationship")

CORRELATION LOADINGS
Shape: (100, 18)
Range: [-0.138, 0.999]

✅ These are what you should use for interpretation!



💡 Interpretation:
  |r| > 0.7: Strong correlation
  |r| < 0.3: Weak correlation
  Sign: Positive/negative relationship


### 4.8 ⭐⭐⭐ **NEW: Correlation Circle Plot**

The **correlation circle** is the STANDARD plot in chemometrics:
- Variables plotted on unit circle
- Distance from origin = importance
- Angle between variables = their correlation
- Standard in all professional software

In [34]:
if optimal_k_pls >= 2:
    fig = go.Figure()
    
    # Plot variables (every 5th for clarity)
    sample_every = 5
    for i in range(0, len(wavelengths), sample_every):
        # Arrow from origin
        fig.add_trace(go.Scatter(
            x=[0, correlation_loadings[i, 0]],
            y=[0, correlation_loadings[i, 1]],
            mode='lines',
            line=dict(color='lightblue', width=1),
            showlegend=False,
            hoverinfo='skip'
        ))
        
        # Point at tip
        fig.add_trace(go.Scatter(
            x=[correlation_loadings[i, 0]],
            y=[correlation_loadings[i, 1]],
            mode='markers',
            marker=dict(size=4, color='blue'),
            showlegend=False,
            text=f'λ={wavelengths[i]:.0f}nm',
            hovertemplate='%{text}<br>r1=%{x:.3f}<br>r2=%{y:.3f}<extra></extra>'
        ))
    
    # Unit circle
    theta = np.linspace(0, 2*np.pi, 100)
    fig.add_trace(go.Scatter(
        x=np.cos(theta),
        y=np.sin(theta),
        mode='lines',
        line=dict(color='red', dash='dash', width=2),
        name='Unit circle'
    ))
    
    # Axes
    fig.add_hline(y=0, line_color="black", line_width=1)
    fig.add_vline(x=0, line_color="black", line_width=1)
    
    fig.update_layout(
        title="⭐ Correlation Circle Plot (Standard in Chemometrics)",
        xaxis_title="Correlation with Component 1",
        yaxis_title="Correlation with Component 2",
        width=700,
        height=700,
        xaxis=dict(scaleanchor="y", scaleratio=1, range=[-1.1, 1.1]),
        yaxis=dict(range=[-1.1, 1.1])
    )
    
    fig.show()
    
    print("\n💡 Interpretation:")
    print("  • Variables far from origin: well represented")
    print("  • Variables close together: positively correlated")
    print("  • Variables opposite: negatively correlated")
    print("  • Variables at 90°: uncorrelated")
else:
    print("⚠️  Need ≥2 components for correlation circle")


💡 Interpretation:
  • Variables far from origin: well represented
  • Variables close together: positively correlated
  • Variables opposite: negatively correlated
  • Variables at 90°: uncorrelated


### 4.9 VIP Scores

In [35]:
def compute_vip(pls_model):
    W = pls_model.x_weights_
    Q = pls_model.y_loadings_
    T = pls_model.x_scores_
    ss = np.sum(T**2, axis=0) * (Q**2).ravel()
    total_ss = np.sum(ss)
    p = W.shape[0]
    vip = np.sqrt(p * np.sum(ss * W**2, axis=1) / total_ss)
    return vip

vip_scores = compute_vip(pls_optimal)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=wavelengths, y=vip_scores,
    mode='lines', fill='tozeroy',
    line=dict(color='darkblue', width=2)
))
fig.add_hline(y=1.0, line_dash="dash", line_color="red",
    annotation_text="VIP = 1 (threshold)")

fig.update_layout(
    title="Variable Importance in Projection (VIP)",
    xaxis_title="Wavelength (nm)",
    yaxis_title="VIP Score",
    width=1000, height=500
)
fig.show()

important_vip = np.sum(vip_scores > 1.0)
print(f"\nWavelengths with VIP > 1: {important_vip}/{len(wavelengths)}")


Wavelengths with VIP > 1: 38/100


### 4.10 ⭐⭐⭐ **NEW: Hotelling T²** - Outlier Detection

Hotelling T² identifies outliers in the PLS score space (X-space).

In [36]:
# Compute Hotelling T²
eigenvalues = pca_full.explained_variance_[:optimal_k_pls]
T2 = np.sum((T_pls / np.sqrt(eigenvalues))**2, axis=1)

# Critical values
n = len(X_std)
k = optimal_k_pls
from scipy.stats import f

F_crit_95 = f.ppf(0.95, k, n-k)
T2_95 = (k * (n-1) / (n-k)) * F_crit_95

F_crit_99 = f.ppf(0.99, k, n-k)
T2_99 = (k * (n-1) / (n-k)) * F_crit_99

print("="*70)
print("HOTELLING T² - OUTLIER DETECTION")
print("="*70)
print(f"\nT² limits:")
print(f"  95%: {T2_95:.2f}")
print(f"  99%: {T2_99:.2f}")
print(f"\nOutliers:")
print(f"  >95%: {np.sum(T2 > T2_95)} samples")
print(f"  >99%: {np.sum(T2 > T2_99)} samples")

# Plot
fig = go.Figure()

colors_t2 = ['green' if t < T2_95 else 'orange' if t < T2_99 else 'red' for t in T2]

fig.add_trace(go.Scatter(
    x=np.arange(len(T2)),
    y=T2,
    mode='markers',
    marker=dict(size=8, color=colors_t2),
    text=[f'Sample {i}<br>T²={T2[i]:.2f}' for i in range(len(T2))],
    hovertemplate='%{text}<extra></extra>'
))

fig.add_hline(y=T2_95, line_dash="dash", line_color="orange",
              annotation_text="95% limit")
fig.add_hline(y=T2_99, line_dash="dash", line_color="red",
              annotation_text="99% limit")

fig.update_layout(
    title="⭐ Hotelling T² - Outliers in Score Space",
    xaxis_title="Sample Index",
    yaxis_title="Hotelling T²",
    width=1000,
    height=500
)

fig.show()

print("\n💡 Interpretation:")
print("  Green: Normal samples")
print("  Orange: Moderate outliers (95-99%)")
print("  Red: Extreme outliers (>99%)")

HOTELLING T² - OUTLIER DETECTION

T² limits:
  95%: 32.39
  99%: 39.64

Outliers:
  >95%: 29 samples
  >99%: 20 samples



💡 Interpretation:
  Green: Normal samples
  Orange: Moderate outliers (95-99%)
  Red: Extreme outliers (>99%)


### 4.11 ⭐⭐⭐ **NEW: Q Residuals (DModX)** - Model Fit

Q residuals measure how well samples are described by the model.

In [37]:
# Compute Q residuals
X_reconstructed = T_pls @ P_pls.T
X_residuals = X_std - X_reconstructed
Q = np.sum(X_residuals**2, axis=1)

# Critical value (approximation)
from scipy.stats import chi2
theta1 = np.mean(Q)
theta2 = np.var(Q)
h0 = 1 - (2*theta1*theta2) / (3*theta2**2) if theta2 > 0 else 1
Q_crit_95 = theta1 * (1 + (chi2.ppf(0.95, 1) * np.sqrt(2*theta2) / theta1) if theta1 > 0 else 1)**h0

print("="*70)
print("Q RESIDUALS (DModX) - MODEL FIT")
print("="*70)
print(f"\nQ critical (95%): {Q_crit_95:.2f}")
print(f"Samples exceeding: {np.sum(Q > Q_crit_95)}")

# Plot
fig = go.Figure()

colors_q = ['green' if q < Q_crit_95 else 'red' for q in Q]

fig.add_trace(go.Scatter(
    x=np.arange(len(Q)),
    y=Q,
    mode='markers',
    marker=dict(size=8, color=colors_q),
    text=[f'Sample {i}<br>Q={Q[i]:.2f}' for i in range(len(Q))],
    hovertemplate='%{text}<extra></extra>'
))

fig.add_hline(y=Q_crit_95, line_dash="dash", line_color="red",
              annotation_text="95% limit (DCrit)")

fig.update_layout(
    title="⭐ Q Residuals (DModX) - Distance to Model",
    xaxis_title="Sample Index",
    yaxis_title="Q Residual",
    width=1000,
    height=500
)

fig.show()

print("\n💡 Interpretation:")
print("  Green: Well described by model")
print("  Red: Poorly described (different structure or errors)")

Q RESIDUALS (DModX) - MODEL FIT

Q critical (95%): 0.00
Samples exceeding: 215



💡 Interpretation:
  Green: Well described by model
  Red: Poorly described (different structure or errors)


### 4.12 ⭐⭐ **NEW: T² vs Q Plot**

Combined view of outlier types.

In [38]:
fig = go.Figure()

# Color by quadrant
colors_combo = []
for t, q in zip(T2, Q):
    if t <= T2_95 and q <= Q_crit_95:
        colors_combo.append('green')  # Normal
    elif t > T2_95 and q <= Q_crit_95:
        colors_combo.append('orange')  # High T² only
    elif t <= T2_95 and q > Q_crit_95:
        colors_combo.append('red')  # High Q only
    else:
        colors_combo.append('darkred')  # Both high!

fig.add_trace(go.Scatter(
    x=T2, y=Q,
    mode='markers',
    marker=dict(size=8, color=colors_combo),
    text=[f'Sample {i}<br>T²={T2[i]:.2f}<br>Q={Q[i]:.2f}' for i in range(len(T2))],
    hovertemplate='%{text}<extra></extra>'
))

fig.add_vline(x=T2_95, line_dash="dash", line_color="red")
fig.add_hline(y=Q_crit_95, line_dash="dash", line_color="red")

# Add quadrant labels
fig.add_annotation(x=T2_95/2, y=Q_crit_95*2, text="High Q<br>(poor fit)",
    showarrow=False, font=dict(size=10))
fig.add_annotation(x=T2_95*2, y=Q_crit_95/2, text="High T²<br>(outlier)",
    showarrow=False, font=dict(size=10))
fig.add_annotation(x=T2_95*2, y=Q_crit_95*2, text="Both<br>(REVIEW!)",
    showarrow=False, font=dict(size=10))

fig.update_layout(
    title="T² vs Q - Outlier Classification",
    xaxis_title="Hotelling T²",
    yaxis_title="Q Residual",
    width=800,
    height=600
)

fig.show()

### 4.13 ⭐⭐⭐ **NEW: Williams Plot** - Applicability Domain

The Williams plot shows prediction reliability.

In [39]:
# Compute leverage
leverage = np.sum(T_pls @ np.linalg.inv(T_pls.T @ T_pls) * T_pls, axis=1)

# Standardized residuals
residuals_cv = y - y_pred_pls_cv
std_residuals = (residuals_cv - np.mean(residuals_cv)) / np.std(residuals_cv)

# Critical values
h_crit = 3 * (optimal_k_pls + 1) / n
r_crit = 3

print("="*70)
print("WILLIAMS PLOT - APPLICABILITY DOMAIN")
print("="*70)
print(f"\nCritical values:")
print(f"  Leverage: {h_crit:.4f}")
print(f"  Residual: ±{r_crit}")
print(f"\nSamples:")
print(f"  High leverage: {np.sum(leverage > h_crit)}")
print(f"  High residual: {np.sum(np.abs(std_residuals) > r_crit)}")

# Plot
fig = go.Figure()

colors_w = []
for h, r in zip(leverage, std_residuals):
    if h <= h_crit and abs(r) <= r_crit:
        colors_w.append('green')
    elif h > h_crit and abs(r) <= r_crit:
        colors_w.append('orange')
    elif h <= h_crit and abs(r) > r_crit:
        colors_w.append('red')
    else:
        colors_w.append('darkred')

fig.add_trace(go.Scatter(
    x=leverage, y=std_residuals,
    mode='markers',
    marker=dict(size=8, color=colors_w),
    text=[f'Sample {i}<br>h={leverage[i]:.3f}<br>r={std_residuals[i]:.2f}' 
          for i in range(len(leverage))],
    hovertemplate='%{text}<extra></extra>'
))

fig.add_vline(x=h_crit, line_dash="dash", line_color="red",
              annotation_text="Leverage limit")
fig.add_hline(y=r_crit, line_dash="dash", line_color="red")
fig.add_hline(y=-r_crit, line_dash="dash", line_color="red")

# Warning zone
fig.add_shape(type="rect",
    x0=h_crit, y0=-r_crit, x1=max(leverage)*1.1, y1=r_crit,
    line=dict(color="orange", width=2, dash="dot"),
    fillcolor="yellow", opacity=0.1
)

fig.update_layout(
    title="⭐ Williams Plot - Prediction Reliability",
    xaxis_title="Leverage (h)",
    yaxis_title="Standardized Residual",
    width=800,
    height=600
)

fig.show()

print("\n💡 Interpretation:")
print("  Green: Reliable predictions")
print("  Orange: High leverage, good fit (influential)")
print("  Red: Outliers (poor fit)")
print("  Dark red: High leverage + outlier (DANGEROUS!)")

WILLIAMS PLOT - APPLICABILITY DOMAIN

Critical values:
  Leverage: 0.2651
  Residual: ±3

Samples:
  High leverage: 10
  High residual: 4



💡 Interpretation:
  Green: Reliable predictions
  Orange: High leverage, good fit (influential)
  Red: Outliers (poor fit)
  Dark red: High leverage + outlier (DANGEROUS!)


### 4.14 ⭐⭐ **NEW: R² Train vs CV** - Overfitting Check

In [40]:
# Compute R² for different k
r2_train_list = []
r2_cv_list = []

for n_comp in range(1, max_components+1):
    pls_temp = PLSRegression(n_components=n_comp, scale=False)
    pls_temp.fit(X_std, y_std)
    y_pred_train = pls_temp.predict(X_std).ravel()
    r2_train_list.append(r2_score(y_std, y_pred_train))
    r2_cv_list.append(1 - pls_results[n_comp]['cv_rmse_mean']**2)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=k_values, y=r2_train_list,
    mode='lines+markers', name='Training R²',
    line=dict(color='blue', width=2)
))

fig.add_trace(go.Scatter(
    x=k_values, y=r2_cv_list,
    mode='lines+markers', name='CV R²',
    line=dict(color='red', width=2)
))

fig.add_vline(x=optimal_k_pls, line_dash="dash", line_color="green",
              annotation_text=f"Optimal k={optimal_k_pls}")

fig.update_layout(
    title="⭐ R² Train vs CV - Overfitting Detection",
    xaxis_title="Number of Components",
    yaxis_title="R²",
    yaxis_range=[0, 1],
    width=1000,
    height=500
)

fig.show()

gap = r2_train_list[optimal_k_pls-1] - r2_cv_list[optimal_k_pls-1]
print(f"\nR² gap at optimal k: {gap:.4f}")
if gap > 0.1:
    print("⚠️  Large gap indicates overfitting!")
else:
    print("✅ Small gap indicates good generalization")


R² gap at optimal k: 0.0198
✅ Small gap indicates good generalization


### 4.15 ⭐⭐ **NEW: Explained Variance (X and Y)**

In [41]:
# Variance in X
var_X_explained = []
for i in range(optimal_k_pls):
    X_recon = T_pls[:, :i+1] @ P_pls[:, :i+1].T
    var_X_explained.append((1 - np.var(X_std - X_recon) / np.var(X_std)) * 100)

# Variance in Y
var_Y_explained = []
for i in range(optimal_k_pls):
    pls_temp = PLSRegression(n_components=i+1, scale=False)
    pls_temp.fit(X_std, y_std)
    y_pred = pls_temp.predict(X_std).ravel()
    var_Y_explained.append(r2_score(y_std, y_pred) * 100)

fig = make_subplots(rows=1, cols=2,
    subplot_titles=("Variance in X", "Variance in Y"))

fig.add_trace(go.Bar(x=list(range(1, optimal_k_pls+1)), y=var_X_explained,
    marker_color='lightblue'), row=1, col=1)

fig.add_trace(go.Bar(x=list(range(1, optimal_k_pls+1)), y=var_Y_explained,
    marker_color='lightcoral'), row=1, col=2)

fig.update_xaxes(title_text="Component", row=1, col=1)
fig.update_xaxes(title_text="Component", row=1, col=2)
fig.update_yaxes(title_text="Variance (%)", row=1, col=1)
fig.update_yaxes(title_text="Variance (%)", row=1, col=2)

fig.update_layout(
    title="⭐ Variance Explained by PLS Components",
    width=1200, height=500,
    showlegend=False
)

fig.show()

print(f"\nTotal variance explained:")
print(f"  X: {var_X_explained[-1]:.2f}%")
print(f"  Y: {var_Y_explained[-1]:.2f}%")


Total variance explained:
  X: 100.00%
  Y: 98.35%


### 4.16 ⭐ **NEW: Q-Q Plot** - Residual Normality

In [42]:
residuals_sorted = np.sort(residuals_cv)
theoretical_q = stats.norm.ppf(np.linspace(0.01, 0.99, len(residuals_cv)))

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=theoretical_q, y=residuals_sorted,
    mode='markers',
    marker=dict(size=6, color='blue')
))

# Reference line
line_x = [theoretical_q.min(), theoretical_q.max()]
line_y = [residuals_sorted.min(), residuals_sorted.max()]
fig.add_trace(go.Scatter(
    x=line_x, y=line_y,
    mode='lines',
    line=dict(color='red', dash='dash', width=2),
    name='Normal'
))

fig.update_layout(
    title="Q-Q Plot - Check Residual Normality",
    xaxis_title="Theoretical Quantiles",
    yaxis_title="Sample Quantiles",
    width=700, height=700
)

fig.show()

shapiro_stat, shapiro_p = stats.shapiro(residuals_cv)
print(f"\nShapiro-Wilk test: p-value = {shapiro_p:.4f}")
if shapiro_p > 0.05:
    print("✅ Residuals appear normally distributed")
else:
    print("⚠️  Residuals may not be normally distributed")


Shapiro-Wilk test: p-value = 0.0000
⚠️  Residuals may not be normally distributed


### 4.17 ⭐⭐ **NEW: Inner Relation Plot** - PLS Validity

In [43]:
if optimal_k_pls >= 2:
    u_pls = pls_optimal.y_scores_
    
    fig = make_subplots(
        rows=1, cols=min(2, optimal_k_pls),
        subplot_titles=[f"Component {i+1}" for i in range(min(2, optimal_k_pls))]
    )
    
    for i in range(min(2, optimal_k_pls)):
        # Scatter
        fig.add_trace(go.Scatter(
            x=T_pls[:, i], y=u_pls[:, i],
            mode='markers',
            marker=dict(size=6, color='blue'),
            showlegend=False
        ), row=1, col=i+1)
        
        # Linear fit
        slope, intercept = np.polyfit(T_pls[:, i], u_pls[:, i], 1)
        x_line = np.array([T_pls[:, i].min(), T_pls[:, i].max()])
        y_line = slope * x_line + intercept
        
        fig.add_trace(go.Scatter(
            x=x_line, y=y_line,
            mode='lines',
            line=dict(color='red', dash='dash', width=2),
            showlegend=False
        ), row=1, col=i+1)
        
        fig.update_xaxes(title_text=f"X-score t{i+1}", row=1, col=i+1)
        fig.update_yaxes(title_text=f"Y-score u{i+1}", row=1, col=i+1)
    
    fig.update_layout(
        title="⭐ Inner Relation - Check PLS Linearity",
        width=1000, height=500
    )
    
    fig.show()
    
    print("💡 Points should follow red line closely")
    print("   R² > 0.9 indicates good linear inner relation")
else:
    print("⚠️  Need ≥2 components")

💡 Points should follow red line closely
   R² > 0.9 indicates good linear inner relation


---

## **Section 5: PLS2 - Multiple Responses**

### 5.1 Create Multiple Response Matrix

In [44]:
np.random.seed(42)
protein = 25 - 0.3 * (y - y.mean()) + np.random.normal(0, 0.5, len(y))
moisture = 70 - 0.4 * (y - y.mean()) + 0.2 * (protein - protein.mean()) + np.random.normal(0, 0.8, len(y))

Y_multi = np.column_stack([y, protein, moisture])
response_names = ['Fat', 'Protein', 'Moisture']

print("="*70)
print("PLS2: MULTIPLE RESPONSES")
print("="*70)
print(f"\nY matrix: {Y_multi.shape}")

for i, name in enumerate(response_names):
    print(f"\n{name}:")
    print(f"  Mean: {Y_multi[:, i].mean():.2f}%")
    print(f"  Range: [{Y_multi[:, i].min():.2f}, {Y_multi[:, i].max():.2f}]")

corr_Y = np.corrcoef(Y_multi.T)
print(f"\nResponse correlations:")
print(pd.DataFrame(corr_Y, index=response_names, columns=response_names).round(3))

PLS2: MULTIPLE RESPONSES

Y matrix: (215, 3)

Fat:
  Mean: 18.14%
  Range: [0.90, 49.10]

Protein:
  Mean: 25.00%
  Range: [14.97, 30.70]

Moisture:
  Mean: 70.05%
  Range: [54.92, 77.94]

Response correlations:
            Fat  Protein  Moisture
Fat       1.000   -0.992    -0.991
Protein  -0.992    1.000     0.985
Moisture -0.991    0.985     1.000


### 5.2 PLS2 Cross-Validation

In [45]:
scaler_Y_multi = StandardScaler()
Y_multi_std = scaler_Y_multi.fit_transform(Y_multi)

max_comp_pls2 = 20
print("Running PLS2 CV...")

pls2_results = {}
for n_comp in range(1, max_comp_pls2+1):
    cv_scores = []
    for train_idx, test_idx in cv.split(X_std):
        X_train, X_test = X_std[train_idx], X_std[test_idx]
        Y_train, Y_test = Y_multi_std[train_idx], Y_multi_std[test_idx]
        pls2 = PLSRegression(n_components=n_comp, scale=False)
        pls2.fit(X_train, Y_train)
        Y_pred = pls2.predict(X_test)
        cv_scores.append(mean_squared_error(Y_test, Y_pred))
    pls2_results[n_comp] = {
        'cv_rmse_mean': np.sqrt(np.mean(cv_scores)),
        'cv_rmse_std': np.std(np.sqrt(cv_scores))
    }

rmse_values_pls2 = [pls2_results[k]['cv_rmse_mean'] for k in range(1, max_comp_pls2+1)]
optimal_k_pls2 = np.argmin(rmse_values_pls2) + 1

print(f"✅ Optimal: k={optimal_k_pls2}, RMSE={rmse_values_pls2[optimal_k_pls2-1]:.4f}")

Running PLS2 CV...
✅ Optimal: k=14, RMSE=0.2150


### 5.3 Fit PLS2 Model

In [46]:
pls2_optimal = PLSRegression(n_components=optimal_k_pls2, scale=False)
pls2_optimal.fit(X_std, Y_multi_std)

Y_pred_pls2_std = pls2_optimal.predict(X_std)
Y_pred_pls2 = scaler_Y_multi.inverse_transform(Y_pred_pls2_std)

# CV predictions
Y_pred_pls2_cv_std = np.zeros_like(Y_multi_std)
for train_idx, test_idx in cv.split(X_std):
    X_train, X_test = X_std[train_idx], X_std[test_idx]
    Y_train, Y_test = Y_multi_std[train_idx], Y_multi_std[test_idx]
    pls2_cv = PLSRegression(n_components=optimal_k_pls2, scale=False)
    pls2_cv.fit(X_train, Y_train)
    Y_pred_pls2_cv_std[test_idx] = pls2_cv.predict(X_test)

Y_pred_pls2_cv = scaler_Y_multi.inverse_transform(Y_pred_pls2_cv_std)

print(f"PLS2 (k={optimal_k_pls2}) CV Performance:")
for i, name in enumerate(response_names):
    r2 = r2_score(Y_multi[:, i], Y_pred_pls2_cv[:, i])
    rmse = np.sqrt(mean_squared_error(Y_multi[:, i], Y_pred_pls2_cv[:, i]))
    print(f"\n{name}:")
    print(f"  R²: {r2:.4f}")
    print(f"  RMSE: {rmse:.4f}%")

PLS2 (k=14) CV Performance:

Fat:
  R²: 0.9623
  RMSE: 2.4665%

Protein:
  R²: 0.9485
  RMSE: 0.8675%

Moisture:
  R²: 0.9505
  RMSE: 1.3047%


### 5.4 PLS2 Predictions

In [47]:
fig = make_subplots(rows=1, cols=3,
    subplot_titles=[f"{name}: Actual vs Predicted (CV)" for name in response_names])

colors = ['blue', 'green', 'orange']

for i, (name, color) in enumerate(zip(response_names, colors)):
    fig.add_trace(go.Scatter(
        x=Y_multi[:, i], y=Y_pred_pls2_cv[:, i],
        mode='markers', marker=dict(color=color, size=6, opacity=0.6),
        showlegend=False), row=1, col=i+1)
    
    y_range = [Y_multi[:, i].min(), Y_multi[:, i].max()]
    fig.add_trace(go.Scatter(
        x=y_range, y=y_range,
        mode='lines', line=dict(color='black', dash='dash'),
        showlegend=False), row=1, col=i+1)
    
    fig.update_xaxes(title_text=f"Actual {name} (%)", row=1, col=i+1)
    fig.update_yaxes(title_text=f"Predicted (%)", row=1, col=i+1)

fig.update_layout(title=f"PLS2 Predictions (k={optimal_k_pls2})",
    width=1400, height=450)
fig.show()

---

## **Section 6: Comprehensive Comparison**

### 6.1 Coefficient Comparison

In [48]:
fig = go.Figure()

fig.add_trace(go.Scatter(x=wavelengths, y=coef_mlr,
    mode='lines', name='OLS',
    line=dict(color='red', width=1), opacity=0.3))

fig.add_trace(go.Scatter(x=wavelengths, y=theta_pcr,
    mode='lines', name=f'PCR (k={optimal_k_pcr})',
    line=dict(color='blue', width=2), opacity=0.6))

fig.add_trace(go.Scatter(x=wavelengths, y=coef_pls,
    mode='lines', name=f'PLS1 (k={optimal_k_pls})',
    line=dict(color='green', width=2.5)))

coef_pls2_fat = pls2_optimal.coef_[:, 0]
fig.add_trace(go.Scatter(x=wavelengths, y=coef_pls2_fat,
    mode='lines', name=f'PLS2 (k={optimal_k_pls2})',
    line=dict(color='purple', width=2.5, dash='dot')))

fig.add_hline(y=0, line_dash="dash", line_color="gray")

fig.update_layout(
    title="Regression Coefficients: All Methods",
    xaxis_title="Wavelength (nm)",
    yaxis_title="Coefficient Value",
    width=1200, height=600,
    hovermode='x unified'
)

fig.show()

print("\n📊 Coefficient L2 Norms:")
print(f"  OLS:  {np.linalg.norm(coef_mlr):.4f}")
print(f"  PCR:  {np.linalg.norm(theta_pcr):.4f}")
print(f"  PLS1: {np.linalg.norm(coef_pls):.4f}")
print(f"  PLS2: {np.linalg.norm(coef_pls2_fat):.4f}")


📊 Coefficient L2 Norms:
  OLS:  6182.9688
  PCR:  407.4074
  PLS1: 417.7805
  PLS2: 6.3921


### 6.2 Performance Comparison

In [49]:
methods = ['OLS', 'PCR', 'PLS1', 'PLS2 (Fat)']
r2_values = [
    cv_r2_mlr.mean(),
    r2_pcr_cv,
    r2_pls_cv,
    r2_score(Y_multi[:, 0], Y_pred_pls2_cv[:, 0])
]
components = [100, optimal_k_pcr, optimal_k_pls, optimal_k_pls2]
rmse_values = [
    cv_rmse_mlr.mean(),
    rmse_pcr_cv,
    rmse_pls_cv,
    np.sqrt(mean_squared_error(Y_multi[:, 0], Y_pred_pls2_cv[:, 0]))
]

fig = go.Figure()

fig.add_trace(go.Bar(
    x=methods, y=r2_values,
    marker_color=['red', 'blue', 'green', 'purple'],
    text=[f'{r2:.4f}' for r2 in r2_values],
    textposition='outside'
))

fig.update_layout(
    title="R² Comparison (Cross-Validation)",
    xaxis_title="Method",
    yaxis_title="R² (CV)",
    yaxis_range=[0, 1],
    width=800, height=500
)

fig.show()

### 6.3 Summary Table

In [50]:
comparison_data = {
    'Method': methods,
    'Components': components,
    'CV R²': r2_values,
    'CV RMSE (%)': rmse_values
}

df_comparison = pd.DataFrame(comparison_data)

print("="*70)
print("FINAL PERFORMANCE COMPARISON")
print("="*70)
print("\n" + df_comparison.to_string(index=False))

print("\n" + "="*70)
print("KEY FINDINGS")
print("="*70)

improvement = (r2_pls_cv - cv_r2_mlr.mean()) * 100

print(f"\n1. OLS FAILURE:")
print(f"   CV R² = {cv_r2_mlr.mean():.4f}")
print(f"   Severe multicollinearity")

print(f"\n2. PCR IMPROVEMENT:")
print(f"   k = {optimal_k_pcr}, R² = {r2_pcr_cv:.4f}")

print(f"\n3. PLS SUPERIORITY:")
print(f"   k = {optimal_k_pls} ({optimal_k_pcr-optimal_k_pls} fewer!)")
print(f"   R² = {r2_pls_cv:.4f} ⭐ BEST")
print(f"   Improvement: {improvement:.1f}%")

print(f"\n4. PLS2 EFFICIENCY:")
print(f"   k = {optimal_k_pls2}")
print(f"   Handles 3 responses efficiently")

FINAL PERFORMANCE COMPARISON

    Method  Components    CV R²  CV RMSE (%)
       OLS         100 0.932984     0.231635
       PCR          30 0.963057     2.443043
      PLS1          18 0.963646     2.423505
PLS2 (Fat)          14 0.962344     2.466526

KEY FINDINGS

1. OLS FAILURE:
   CV R² = 0.9330
   Severe multicollinearity

2. PCR IMPROVEMENT:
   k = 30, R² = 0.9631

3. PLS SUPERIORITY:
   k = 18 (12 fewer!)
   R² = 0.9636 ⭐ BEST
   Improvement: 3.1%

4. PLS2 EFFICIENCY:
   k = 14
   Handles 3 responses efficiently


---

## **🎉 Tutorial Complete!**

### **What You've Learned**

✅ **Data Exploration** (5 plots)
- Multicollinearity assessment
- Response distribution
- Wavelength correlations

✅ **OLS Analysis** (4 plots)
- Why it fails
- Coefficient instability
- Overfitting diagnostics

✅ **PCR Analysis** (4 plots)
- PCA variance explained
- PC-response correlations
- Cross-validation

✅ **PLS Comprehensive** (18 plots!) ⭐
- Score plots (2D & 3D)
- Standard loadings
- **Correlation loadings** (NEW!)
- **Correlation circle** (NEW!)
- VIP scores
- **Hotelling T²** (NEW!)
- **Q residuals / DModX** (NEW!)
- **T² vs Q plot** (NEW!)
- **Williams plot** (NEW!)
- **R² train vs CV** (NEW!)
- **Explained variance X & Y** (NEW!)
- **Q-Q plot** (NEW!)
- **Inner relation** (NEW!)

✅ **PLS2 Analysis** (3 plots)
- Multiple response modeling
- Efficiency demonstration

✅ **Final Comparison** (3 plots)
- All methods side-by-side
- Performance metrics

**Total: ~35 Interactive Plots!** 🎨

---

### **Key Takeaways**

1. **OLS fails** with multicollinearity → Use dimension reduction
2. **PCR works** but is unsupervised → May need many components
3. **PLS wins!** Supervised, fewer components, best accuracy
4. **Correlation loadings** are INDUSTRY STANDARD for interpretation
5. **Hotelling T² & Q residuals** essential for outlier detection
6. **Williams plot** defines applicability domain
7. **PLS2** efficiently handles multiple correlated responses

---

### **When to Use Each Method**

**OLS:** Low correlation, n >> p  
**PCR:** Need X-structure interpretation  
**PLS1:** High multicollinearity, single response ⭐ RECOMMENDED  
**PLS2:** Multiple correlated responses  

---

### **References**

1. Wold et al. (1983) - Original PLS
2. Geladi & Kowalski (1986) - PLS Tutorial
3. Martens & Næs (1989) - Multivariate Calibration
4. Eriksson et al. (2013) - Multi- and Megavariate Analysis

---

**Course:** TTK4260 - Multivariate Data Analysis and ML  
**Instructor:** Adil Rasheed, NTNU

**Questions?** adil.rasheed@ntnu.no

---

## 🚀 **What's Next?**

- Apply to your own data
- Try advanced variants (OPLS, Sparse PLS)
- Explore other methods (Ridge, Lasso, Elastic Net)
- Read the chemometrics literature

**Happy Learning!** 🎓